# Revenue model benchmark

Compare a human-selected linear trend with a model selected by expanding-window validation. Both final models use the same training observations and untouched chronological test set. Re-running the cells recalculates every score.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

root = Path.cwd()
if not (root / "datasets").exists(): root = root.parent
df = pd.read_csv(root / "datasets" / "autods_demo_sales.csv", parse_dates=["Order Date"])
series = (df.dropna(subset=["Order Date", "Revenue"]).set_index("Order Date")["Revenue"]
          .resample("W").sum().astype(float))
if len(series) < 20:
    raise ValueError(f"Need at least 20 weekly observations for validation and holdout; found {len(series)}")
test_n = max(4, min(round(len(series) * 0.2), len(series) - 16))
train, test = series.iloc[:-test_n], series.iloc[-test_n:]
print(f"Train: {train.index.min().date()} through {train.index.max().date()}; test: {test.index.min().date()} through {test.index.max().date()}")

In [ ]:
def fit_predict(name, history, steps):
    x = np.arange(len(history)).reshape(-1, 1)
    future = np.arange(len(history), len(history) + steps).reshape(-1, 1)
    if name == "moving_average":
        return np.repeat(history.tail(min(4, len(history))).mean(), steps)
    model = LinearRegression() if name == "linear_regression" else RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42)
    model.fit(x, history.to_numpy())
    return np.clip(model.predict(future), 0, None)

def mae_for(name, history, actual):
    return mean_absolute_error(actual.to_numpy(), fit_predict(name, history, len(actual)))

validation = []
for name in ["linear_regression", "random_forest", "moving_average"]:
    fold_errors = []
    for fraction in (0.65, 0.8, 0.9):
        cut = int(len(train) * fraction)
        if cut < 8 or len(train) - cut < 2: continue
        fold_errors.append(mae_for(name, train.iloc[:cut], train.iloc[cut:]))
    if fold_errors:
        validation.append({"model": name, "validation_mae": float(np.mean(fold_errors)), "folds": len(fold_errors)})
validation = pd.DataFrame(validation).sort_values("validation_mae")
display(validation)
ai_selected = validation.iloc[0]["model"]
human_selected = "linear_regression"
print("Validation-selected model:", ai_selected)

In [ ]:
def test_metrics(name):
    predicted = fit_predict(name, train, len(test))
    actual = test.to_numpy()
    nonzero = np.abs(actual) > 1e-9
    return {
        "model": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "MAPE_pct": np.mean(np.abs((actual[nonzero] - predicted[nonzero]) / actual[nonzero])) * 100 if nonzero.any() else np.nan,
        "R2": r2_score(actual, predicted) if len(actual) > 1 else np.nan,
    }

comparison = pd.DataFrame([
    {"selection": "Human-selected", **test_metrics(human_selected)},
    {"selection": "Validation-selected", **test_metrics(ai_selected)},
])
display(comparison)
comparison.set_index("selection")[["MAE", "RMSE"]].plot.bar(title="Held-out weekly revenue errors")